# Dashboard Construction

Turns county table & team's model outputs into **one self contained file, `dashboard.html`**. It has no server and no map library: the counties are projected and simplified here with GeoPandas and written into the page as SVG, so the file opens by double-click and works offline.

**Reads**

| File | From | Columns |
|---|---|---|
| `data/clean/county_table.csv`, `sites.csv`, `projected.csv`, `counties.gpkg` | notebooks 01 to 06 | as written |
| `data/model/predictions.csv` | supervised models | `fips`, `prob_base`, `prob_state`, `prob_aug` (out-of-fold, calibrated), `driver_1`, `driver_2`, `driver_3` (short plain-language text) |
| `data/model/clusters.csv` | clustering | `fips`, `cluster` (0, 1, 2 ...), `cluster_name`, `entropy` (0 = certain, 1 = split evenly) |
| `data/model/metrics.json` | supervised models | `{"model": "...", "finding": "one sentence", "scores": [{"features": "...", "pr_auc": 0.0, "brier": 0.0}, ...]}` with the policy row last and the state-identity row just before it |

Any model file that is missing is replaced by a **placeholder** so the layout can be reviewed early. The page says so in a banner at the top, and the banner disappears once all three real files exist.

**Writes** `dashboard.html`.

In [ ]:
import datetime as dt
import json

import geopandas as gpd
import numpy as np
import pandas as pd
import shapely

from common import CLEAN, ROOT, load_counties, read_clean

MODEL = ROOT / "data" / "model"
MODEL.mkdir(parents=True, exist_ok=True)
EQUAL_AREA = "ESRI:102003"
MAP_W = 9600            # map width in drawing units; coordinates are whole numbers at this scale
SIMPLIFY_M = 1500       # boundary detail dropped below this many metres

## County shapes as SVG paths

`coverage_simplify` simplifies shared borders once, so neighbouring counties still meet with no gaps or overlaps.

In [ ]:
table = read_clean("county_table").sort_values("fips").reset_index(drop=True)
geo = load_counties(conus_only=True).to_crs(EQUAL_AREA).set_index("fips").loc[table["fips"]].reset_index()
try:
    geo["geometry"] = shapely.coverage_simplify(geo.geometry.values, tolerance=SIMPLIFY_M)
except Exception as e:
    print("coverage_simplify unavailable, falling back to per-county simplify:", type(e).__name__)
    geo["geometry"] = geo.geometry.simplify(SIMPLIFY_M)

minx, miny, maxx, maxy = geo.total_bounds
k = MAP_W / (maxx - minx)
MAP_H = int(np.ceil((maxy - miny) * k))
to_xy = lambda x, y: (np.rint((np.asarray(x) - minx) * k).astype(int), np.rint((maxy - np.asarray(y)) * k).astype(int))


def ring_path(coords):
    x, y = to_xy(*np.asarray(coords)[:, :2].T)
    keep = np.r_[True, (np.diff(x) != 0) | (np.diff(y) != 0)]
    x, y = x[keep], y[keep]
    if len(x) < 4:
        return ""
    return f"M{x[0]} {y[0]}l" + " ".join(f"{a} {b}" for a, b in zip(np.diff(x)[:-1], np.diff(y)[:-1])) + "z"


def geom_path(g, outline_only=False):
    out = []
    for poly in getattr(g, "geoms", [g]):
        if poly.geom_type != "Polygon":
            continue
        out.append(ring_path(poly.exterior.coords))
        if not outline_only:
            out += [ring_path(r.coords) for r in poly.interiors]
    return "".join(out)


paths = [geom_path(g) for g in geo.geometry]
states_path = "".join(geom_path(g, outline_only=True) for g in geo.dissolve("state_abb").geometry)
cx, cy = to_xy(geo.geometry.representative_point().x, geo.geometry.representative_point().y)
print(f"{len(paths):,} counties, map {MAP_W} x {MAP_H}, {sum(map(len, paths)) / 1e6:.2f} MB of path data")
assert all(paths), "a county lost its shape in simplification: lower SIMPLIFY_M"


def project(df):
    """lat/lon columns -> flat [x0, y0, x1, y1, ...] in map units."""
    if not len(df):
        return []
    p = gpd.GeoSeries(gpd.points_from_xy(df["lon"], df["lat"]), crs=4326).to_crs(EQUAL_AREA)
    return np.c_[to_xy(p.x, p.y)].ravel().tolist()


sites = read_clean("sites")
proj_all = read_clean("projected")
proj_cols = read_clean("projected_county_scenario")
ref = proj_all[proj_all["reference"]]   # only the reference scenario is drawn on the map
n_scen = proj_cols.groupby(["growth_scenario", "market_gravity_weight"]).ngroups

## Model outputs, or placeholders

The placeholders are built to look like plausible output, not random noise, so the layout is judged on something map-shaped: likelihood rises near existing facilities, each state gets its own offset, and states with an enacted data center bill get a policy bump. **None of it is a result.**

In [ ]:
placeholder = []
rng = np.random.default_rng(696)
sig = lambda z: 1 / (1 + np.exp(-z))

pred_file, clus_file, met_file = MODEL / "predictions.csv", MODEL / "clusters.csv", MODEL / "metrics.json"

if pred_file.exists():
    pred = pd.read_csv(pred_file, dtype={"fips": str})
else:
    placeholder.append("Model predictions are simulated.")
    cen = geo.geometry.centroid
    sp = gpd.GeoSeries(gpd.points_from_xy(sites["lon"], sites["lat"]), crs=4326).to_crs(EQUAL_AREA)
    d2 = (cen.x.values[:, None] - sp.x.values[None, :]) ** 2 + (cen.y.values[:, None] - sp.y.values[None, :]) ** 2
    near = np.log1p(np.exp(-d2 / (2 * 90_000.0 ** 2)).sum(1))
    z = (near - near.mean()) / (near.std() or 1)
    suitable = table["is_suitable_county"].fillna(False).astype(bool).values
    base = -2.6 + 1.5 * z + 0.4 * suitable
    st = table["state_abb"]
    state_fx = st.map({s: rng.normal(0, 0.35) for s in st.unique()}).values
    law = (table["law_dc_strict_enacted"].fillna(0) > 0).values
    policy_fx = st.map({s: rng.normal(0, 0.25) for s in st.unique()}).values + 0.5 * law
    names = np.array(["Close to existing data centers", "Far from existing data centers", "Has land suitable for a new site",
                      "Little suitable land", "State enacted a data center bill", "No enacted data center bill in the state",
                      "Low industrial electricity rate", "Short distance to a substation"])
    pick = np.c_[np.where(z > 0, 0, 1), np.where(suitable, 2, 3), np.where(law, 4, 5)]
    pred = pd.DataFrame({"fips": table["fips"], "prob_base": sig(base), "prob_state": sig(base + state_fx),
                         "prob_aug": sig(base + state_fx + policy_fx),
                         "driver_1": names[pick[:, 0]], "driver_2": names[pick[:, 1]], "driver_3": names[pick[:, 2]]})

if clus_file.exists():
    clus = pd.read_csv(clus_file, dtype={"fips": str})
else:
    placeholder.append("Archetypes are simulated.")
    p0 = pred.set_index("fips").loc[table["fips"], "prob_base"].values
    suit = table["percent_county_area_suitable"].fillna(0).values
    c = np.where(p0 > np.quantile(p0, 0.8), 0, np.where(suit > np.median(suit), 1, 2))
    clus = pd.DataFrame({"fips": table["fips"], "cluster": c,
                         "cluster_name": np.array(["Archetype A", "Archetype B", "Archetype C"])[c],
                         "entropy": np.clip(rng.beta(1.5, 4, len(table)), 0, 1)})

d = table.merge(pred, on="fips", how="left").merge(clus, on="fips", how="left")
assert len(d) == len(table), "predictions.csv or clusters.csv has repeated fips codes"
need = ["prob_base", "prob_state", "prob_aug", "cluster"]
assert d[need].notna().all().all(), f"counties missing model output:\n{d[need].isna().sum()}"
y = d["has_center"].astype(int).values


def pr_auc(y, p):
    """Average precision, the same quantity as sklearn's average_precision_score."""
    o = np.argsort(-p, kind="stable")
    hit = y[o]
    return float((np.cumsum(hit) / np.arange(1, len(y) + 1))[hit == 1].mean()) if hit.sum() else float("nan")


if met_file.exists():
    metrics = json.loads(met_file.read_text())
else:
    placeholder.append("Scores and the headline are simulated.")
    metrics = dict(model="Placeholder model", finding="The one-sentence finding goes here once the models have run.",
                   scores=[dict(features=f, pr_auc=pr_auc(y, d[c].values), brier=float(((d[c] - y) ** 2).mean()))
                           for f, c in (("Infrastructure only", "prob_base"), ("Infrastructure + state identity", "prob_state"),
                                        ("Infrastructure + policy features", "prob_aug"))])
print("placeholders in use:", placeholder or "none")
pd.DataFrame(metrics["scores"]).round(3)

## Write the page

In [ ]:
clusters = (d.groupby("cluster").agg(name=("cluster_name", "first"), n=("fips", "size"), share=("has_center", "mean"))
            .sort_index().reset_index(drop=True))
cl_index = d["cluster"].map({c: i for i, c in enumerate(sorted(d["cluster"].unique()))})
drivers = d[["driver_1", "driver_2", "driver_3"]].fillna("").astype(str)
vocab = sorted(set(drivers.values.ravel()) - {""})
vid = {v: i for i, v in enumerate(vocab)}
opt = lambda s, nd=None: [None if pd.isna(v) else (round(float(v), nd) if nd is not None else int(v)) for v in s]
manifest = json.loads((CLEAN / "manifest.json").read_text())

data = dict(
    W=MAP_W, H=MAP_H, states=states_path, sites=project(sites), proj=project(ref), nscen=int(n_scen),
    fips=d["fips"].tolist(), name=d["name"].tolist(), st=d["state_abb"].tolist(), path=paths, cx=cx.tolist(), cy=cy.tolist(),
    nc=opt(d["n_centers"]), big=opt(d["n_large"]), suit=opt(d["percent_county_area_suitable"], 1),
    via=d["is_suitable_county"].fillna(False).astype(int).tolist(),
    pb=opt(d["prob_base"], 4), ps=opt(d["prob_state"], 4), pa=opt(d["prob_aug"], 4),
    cl=cl_index.tolist(), ent=opt(d["entropy"].fillna(0).clip(0, 1), 2),
    pm=d["pnnl_modeled"].astype(int).tolist(), pn=opt(d["n_scenarios_sited"].fillna(0)), pr=opt(d["n_projected"].fillna(0)),
    drv=[[vid[v] for v in row if v] for row in drivers.values], dv=vocab,
    clusters=[dict(name=r["name"], n=int(r["n"]), share=round(float(r["share"]), 4)) for _, r in clusters.iterrows()],
    metrics=metrics, placeholder=placeholder, built=f"{dt.date.today():%B} {dt.date.today().day}, {dt.date.today().year}",
    sources=[manifest[kk]["source"] for kk in ("atlas_county", "projected_county", "cerf_county", "ncsl_state_year", "fips_reference")],
)
body = (ROOT / "dashboard_template.html").read_text().replace("/*DATA*/null/*END*/", json.dumps(data, separators=(",", ":"), allow_nan=False))
page = ('<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">'
        "<style>body{margin:0}[hidden]{display:none!important}img{max-width:100%}</style></head><body>" + body + "</body></html>")
(ROOT / "dashboard.html").write_text(page, encoding="utf-8")
(ROOT / "dashboard_body.html").write_text(body, encoding="utf-8")   # same page without the outer wrapper, for publishing
print(f"wrote dashboard.html ({len(page) / 1e6:.1f} MB): {len(d):,} counties, {len(sites):,} facilities, {len(ref):,} PNNL sites, "
      f"{len(clusters)} archetypes")